# EduTech Global · 05 · Constraint Management (Goal Seek)

**Task 4.** Determine the **minimum number of enterprise licenses** EduTech must sell to offset a 5% rise in domestic interest rates.

**Logic:** Higher interest rates raise the cost of capital → raise the hurdle rate (required return) → raise the revenue/contribution margin needed to break even. Goal Seek (Excel native: Data → What-If Analysis → Goal Seek) finds the license count that makes NPV ≥ 0.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:


from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "edutech":
            return parent
        if (parent / "scripts").exists() and (parent / "outputs").exists() and (parent / "data").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Allow either dataset/ or data/ to hold the source CSV; accept both filenames
HR_CSV_NAMES = ["WA_Fn-UseC_-HR-Employee-Attrition.csv", "HR-Employee-Attrition.csv"]
hr_locations = []
for name in HR_CSV_NAMES:
    hr_locations.extend([DATASET_DIR / name, DATA_DIR / name])
HR_CSV_PATH = next((p for p in hr_locations if p.exists()), None)
assert HR_CSV_PATH is not None, (
    f"Could not find HR CSV. Looked for {HR_CSV_NAMES} in {DATASET_DIR} and {DATA_DIR}"
)
print(f"HR data file : {HR_CSV_PATH}")


In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, Reference

wb = Workbook()
ARIAL = lambda **kw: Font(name="Arial", **kw)
header_fill = PatternFill("solid", fgColor="1F3864")
input_fill  = PatternFill("solid", fgColor="FFF2CC")
result_fill = PatternFill("solid", fgColor="E2EFDA")
band_fill   = PatternFill("solid", fgColor="F2F2F2")
thin = Side(border_style="thin", color="BFBFBF")
box  = Border(left=thin, right=thin, top=thin, bottom=thin)

# === Sheet 1: README ===
readme = wb.active
readme.title = "README"
readme["A1"] = "EduTech Global · License Break-Even Calculator"
readme["A1"].font = ARIAL(size=18, bold=True, color="1F3864")
readme["A2"] = "Task 4 — Goal Seek for minimum licenses to offset 5% rate hike"
readme["A2"].font = ARIAL(size=11, italic=True, color="595959")

readme["A4"] = "PURPOSE"; readme["A4"].font = ARIAL(size=12, bold=True)
readme["A5"] = ("Determine the minimum number of enterprise licenses EduTech must sell so that, "
                "after a 5 percentage-point rise in the domestic interest rate, the project still "
                "delivers a positive NPV over the 5-year horizon.")

readme["A7"] = "HOW TO USE GOAL SEEK"
readme["A7"].font = ARIAL(size=12, bold=True)
gs_steps = [
    "1. Open the 'Calculator' sheet.",
    "2. Note the current 'NPV (5-yr)' result — it should be positive at default inputs.",
    "3. Now apply the rate shock: change cell B6 (Interest rate %) to 5% above the base.",
    "4. NPV will turn negative.",
    "5. Go to Data → What-If Analysis → Goal Seek.",
    "6. Set cell  : B14 (NPV)",
    "7. To value : 0",
    "8. By changing cell : B5 (Licenses sold per year)",
    "9. Click OK. Excel will iterate to find the break-even license count.",
    "",
    "Alternatively, the 'Sensitivity' sheet shows NPV across a license × rate grid (no Goal Seek needed).",
]
for i, s in enumerate(gs_steps, start=8):
    readme[f"A{i}"] = s

readme["A22"] = "NOTE"
readme["A22"].font = ARIAL(size=12, bold=True)
readme["A23"] = ("This file uses standard NPV math. The break-even license count is the lever; the "
                 "interest rate hike is the constraint. Goal Seek runs Excel's iterative solver behind "
                 "the scenes.")
readme.column_dimensions["A"].width = 90

print("README sheet built")


In [ ]:
# === Sheet 2: Calculator ===
calc = wb.create_sheet("Calculator")
calc["A1"] = "BREAK-EVEN LICENSE CALCULATOR"
calc["A1"].font = ARIAL(size=16, bold=True, color="FFFFFF")
calc["A1"].fill = header_fill
calc["A1"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
calc.row_dimensions[1].height = 26
calc.merge_cells("A1:E1")

# INPUTS
calc["A3"] = "INPUTS (yellow cells)"
calc["A3"].font = ARIAL(size=12, bold=True, color="1F3864")

inputs = [
    ("Annual contribution margin per license (USD)",  400,    "$#,##0"),    # B4
    ("Licenses sold per year (Goal Seek changes this)", 3000, "#,##0"),    # B5
    ("Interest rate (discount rate, %)",                0.10, "0.00%"),    # B6 — base 10%
    ("Project horizon (years)",                         5,    "0"),        # B7
    ("One-time entry cost (USD)",                       2_400_000, "$#,##0"),  # B8
    ("Annual fixed operating cost (USD)",               900_000,   "$#,##0"),  # B9
    ("Annual license growth rate (%)",                  0.15, "0.0%"),     # B10
]
for i, (lbl, val, fmt) in enumerate(inputs, start=4):
    calc[f"A{i}"] = lbl; calc[f"B{i}"] = val
    calc[f"A{i}"].font = ARIAL(size=10)
    calc[f"B{i}"].font = ARIAL(size=10, bold=True, color="0000FF")
    calc[f"B{i}"].fill = input_fill; calc[f"B{i}"].border = box
    calc[f"B{i}"].number_format = fmt

# Year-by-year cash flow table
calc["A13"] = "YEAR-BY-YEAR CASH FLOW"
calc["A13"].font = ARIAL(size=12, bold=True, color="1F3864")

# Header
yr_headers = ["Year", "Licenses", "Revenue", "Op cost", "Net cash flow", "Discount factor", "PV"]
for j, h in enumerate(yr_headers, start=1):
    c = calc.cell(row=15, column=j, value=h)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill; c.border = box
    c.alignment = Alignment(horizontal="center")

# Year 0
calc.cell(row=16, column=1, value=0).number_format = "0"
calc.cell(row=16, column=2, value=0).number_format = "#,##0"
calc.cell(row=16, column=3, value=0).number_format = "$#,##0"
calc.cell(row=16, column=4, value="=-$B$8").number_format = "$#,##0"      # entry cost
calc.cell(row=16, column=5, value="=C16+D16").number_format = "$#,##0"
calc.cell(row=16, column=6, value=1).number_format = "0.000"
calc.cell(row=16, column=7, value="=E16*F16").number_format = "$#,##0"

# Years 1..N (we hard-code 5 rows, only those within horizon contribute)
for yr in range(1, 6):
    r = 16 + yr
    # Year number
    calc.cell(row=r, column=1, value=yr).number_format = "0"
    # Licenses sold this year (with growth)
    calc.cell(row=r, column=2,
              value=f"=IF($B$7>={yr},$B$5*(1+$B$10)^({yr}-1),0)").number_format = "#,##0"
    # Revenue (=contribution margin * licenses)
    calc.cell(row=r, column=3, value=f"=$B$4*B{r}").number_format = "$#,##0"
    # Op cost
    calc.cell(row=r, column=4,
              value=f"=IF($B$7>={yr},-$B$9,0)").number_format = "$#,##0"
    # Net cash flow
    calc.cell(row=r, column=5, value=f"=C{r}+D{r}").number_format = "$#,##0"
    # Discount factor
    calc.cell(row=r, column=6, value=f"=1/(1+$B$6)^{yr}").number_format = "0.000"
    # Present value
    calc.cell(row=r, column=7, value=f"=E{r}*F{r}").number_format = "$#,##0"

# NPV row
calc.cell(row=22, column=1, value="NPV (5-yr)").font = ARIAL(size=12, bold=True)
calc.cell(row=22, column=7, value="=SUM(G16:G21)")
calc.cell(row=22, column=7).number_format = "$#,##0"
calc.cell(row=22, column=7).font = ARIAL(size=12, bold=True, color="00703C")
calc.cell(row=22, column=7).fill = result_fill

# IMPORTANT: B14 is referenced in README as the NPV cell for Goal Seek.
# Let's also stamp the NPV at B14 explicitly so README instructions are accurate.
calc["A14"] = "→ NPV (5-yr, USD)"
calc["A14"].font = ARIAL(size=12, bold=True)
calc["B14"] = "=G22"
calc["B14"].number_format = "$#,##0"
calc["B14"].font = ARIAL(size=14, bold=True, color="00703C")
calc["B14"].fill = result_fill

# Borders + alternating bands
for r in range(15, 23):
    for c in range(1, 8):
        calc.cell(row=r, column=c).border = box

# Column widths
calc.column_dimensions["A"].width = 38
calc.column_dimensions["B"].width = 18
for col in "CDEFG":
    calc.column_dimensions[col].width = 16

print("Calculator sheet built")


In [ ]:
# === Sheet 3: Sensitivity grid ===
sens = wb.create_sheet("Sensitivity")
sens["A1"] = "NPV SENSITIVITY — Licenses sold × Interest rate"
sens["A1"].font = ARIAL(size=14, bold=True, color="1F3864")
sens.merge_cells("A1:H1")

sens["A3"] = "Licenses ↓ \ Rate →"
sens["A3"].font = ARIAL(size=10, bold=True)

# X-axis: interest rates from current to +5pp
rates = [0.10, 0.115, 0.13, 0.145, 0.15, 0.16]
for j, rt in enumerate(rates):
    sens.cell(row=3, column=2+j, value=rt).number_format = "0.0%"
    sens.cell(row=3, column=2+j).font = ARIAL(size=10, bold=True)

# Y-axis: license counts
license_counts = [2000, 2500, 2750, 3000, 3100, 3200, 3300, 3500, 4000, 5000]

# For each (licenses, rate), compute NPV using the formula structure
# NPV = -Entry + sum_y=1..5 ((CM*L*(1+g)^(y-1) - FixedCost) / (1+r)^y)
# We'll write this as a single Excel formula referencing the Calculator inputs
# B4=CM, B7=horizon, B8=entry, B9=fixed, B10=growth
def npv_formula(L, r, growth_factor=True):
    parts = ["-Calculator!$B$8"]
    for y in range(1, 6):
        # cf_y = CM*L*(1+g)^(y-1) - fixed
        # discount = 1/(1+r)^y
        cf = f"((Calculator!$B$4*{L}*(1+Calculator!$B$10)^({y}-1)-Calculator!$B$9)/(1+{r})^{y})"
        parts.append(cf)
    return "=" + "+".join(parts)

for i, L in enumerate(license_counts):
    sens.cell(row=4+i, column=1, value=L).number_format = "#,##0"
    sens.cell(row=4+i, column=1).font = ARIAL(size=10, bold=True)
    for j, r in enumerate(rates):
        c = sens.cell(row=4+i, column=2+j, value=npv_formula(L, r))
        c.number_format = "$#,##0;[Red]-$#,##0"
        c.font = ARIAL(size=9)

# Notes
sens["A15"] = "Reading: GREEN/POSITIVE values = profitable; RED/NEGATIVE = losses."
sens["A15"].font = ARIAL(size=9, italic=True, color="595959")
sens["A16"] = "The break-even contour (NPV = 0) shows the minimum licenses needed at each interest rate."
sens["A16"].font = ARIAL(size=9, italic=True, color="595959")
sens.merge_cells("A15:H15"); sens.merge_cells("A16:H16")

sens.column_dimensions["A"].width = 22
for col in range(2, 9):
    sens.column_dimensions[chr(ord("A") + col - 1)].width = 14

# SAVE
out_file = OUTPUTS_DIR / "License_BreakEven_GoalSeek.xlsx"
wb.save(out_file)
print(f"Saved: {out_file}")


## Sanity-check the math in Python

The break-even license count at +5% rate hike — what should we expect?

In [ ]:
def npv(licenses, rate, cm=400, horizon=5, entry=2_400_000, fixed=900_000, growth=0.15):
    pv = -entry
    for y in range(1, horizon+1):
        cf = cm * licenses * (1+growth)**(y-1) - fixed
        pv += cf / (1+rate)**y
    return pv

base_rate = 0.10
shocked_rate = 0.15

# At base rate, default 3000 licenses
print(f"At base rate ({base_rate:.0%}), 3,000 licenses NPV: ${npv(3000, base_rate):,.0f}")
print(f"At shocked rate ({shocked_rate:.0%}), 3,000 licenses NPV: ${npv(3000, shocked_rate):,.0f}")

# Find break-even at shocked rate
from scipy.optimize import brentq
breakeven = brentq(lambda L: npv(L, shocked_rate), 100, 100_000)
print(f"\n🎯 Break-even at shocked rate ({shocked_rate:.0%}): {breakeven:,.0f} licenses")
print(f"   That's a {(breakeven/3000-1)*100:+.1f}% increase over the base-case license target.")
print(f"\nIn Excel, run Goal Seek: Set B14 to value 0 by changing B5. You should get ~{breakeven:,.0f}.")


✅ **Notebook 05 complete.** Move to `06_build_dashboard_xlsx.ipynb`.